In [9]:
!apt-get update -y
!apt-get install -y zstd
!which zstd
!zstd --version
!dpkg -l | grep zstd || true

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 https://cli.github.com/packages stable InRelease
Hit:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [10]:
!pip -q install ollama gradio pillow langgraph langchain-core langgraph.checkpoint.sqlite
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [17]:
import subprocess, os, time

!pkill -f ollama 2>/dev/null; sleep 2

env = {
    **os.environ,
    "OLLAMA_NUM_GPU": "999",
    "CUDA_VISIBLE_DEVICES": "0",
    "LD_LIBRARY_PATH": (
        "/usr/local/cuda-12.8/compat:"
        "/usr/lib64-nvidia:"
        "/usr/local/cuda/lib64:"
        "/usr/local/cuda/targets/x86_64-linux/lib:"
        + os.environ.get("LD_LIBRARY_PATH", "")
    ),
}

log = open("/tmp/ollama_server.log", "w")
subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=log)
time.sleep(5)

# Confirm GPU is found
!cat /tmp/ollama_server.log | grep -i "library\|error"

^C
time=2026-03-10T03:15:43.817Z level=INFO source=routes.go:1658 msg="server config" env="map[CUDA_VISIBLE_DEVICES:0 GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/root/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* file://* tauri://* vscode-webview://* vscode-file://*] OLLAM

In [18]:
!ollama pull llava:7b-v1.5-q4_0

In [19]:
!ollama list

NAME                  ID              SIZE      MODIFIED     
llava:7b-v1.5-q4_0    cd3274b81a85    4.5 GB    1 second ago    


In [ ]:
import ollama
ollama.chat(model="llava:7b-v1.5-q4_0", messages=[{"role":"user","content":"hello"}])


ChatResponse(model='llava:7b-v1.5-q4_0', created_at='2026-02-26T16:36:47.250977992Z', done=True, done_reason='stop', total_duration=35510633380, load_duration=28251804, prompt_eval_count=13, prompt_eval_duration=746937675, eval_count=45, eval_duration=34701597862, message=Message(role='assistant', content='喵喵，你好呀！很高兴见到你。请问有什么我能为你效劳的吗？', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

In [ ]:
import ollama, base64

# Load and encode a tiny test image directly
with open("/content/Felis_silvestris_silvestris_small_gradual_decrease_of_quality_-_JPEG_compression.jpg", "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode("utf-8")

print("sending image to llava...")

response = ollama.chat(
    model="llava:7b-v1.5-q4_0",
    messages=[{
        "role": "user",
        "content": "What is in this image?",
        "images": [img_b64]
    }]
)
print(response)

sending image to llava...
model='llava:7b-v1.5-q4_0' created_at='2026-02-24T16:44:42.392737702Z' done=True done_reason='stop' total_duration=280815607102 load_duration=33469206 prompt_eval_count=596 prompt_eval_duration=203337647447 eval_count=69 eval_duration=62418038924 message=Message(role='assistant', content=" In the image, there is a cat sitting on snow. The cat appears to be staring off into the distance and seems focused on something beyond the camera's frame of view. The color tones in the photo are muted, giving it an overall monochromatic look with shades of green, brown, and orange.", thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None


## Exercise 1

In [ ]:
import base64
from typing import TypedDict, Annotated, Sequence
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, AnyMessage

import ollama
from IPython.display import display
import ipywidgets as widgets
from PIL import Image
import io
from google.colab import files


class AgentState(TypedDict):
    messages: Annotated[Sequence[AnyMessage], add_messages]
    user_input: str
    image_b64: str | None
    should_exit: bool

from PIL import Image
import io, base64

def prepare_image_for_llava(image_bytes,
                            max_dim=128,
                            max_megapixels=0.1,
                            jpeg_quality=70):
    """
    Resize + compress image so Ollama LLaVA won't crash.
    Returns base64 string.
    """

    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")

    # ---------- Step 1: limit megapixels ----------
    w, h = img.size
    megapixels = (w*h)/1_000_000

    if megapixels > max_megapixels:
        scale = (max_megapixels/megapixels) ** 0.5
        img = img.resize(
            (int(w*scale), int(h*scale)),
            Image.LANCZOS
        )

    # ---------- Step 2: limit dimension ----------
    img.thumbnail((max_dim, max_dim), Image.LANCZOS)

    # ---------- Step 3: compress JPEG ----------
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=jpeg_quality, optimize=True)

    # ---------- Step 4: convert to base64 ----------
    img_b64 = base64.b64encode(buffer.getvalue()).decode()

    print(f"Final image size: {img.size}")

    return img_b64


def upload_image(state):

    print("\nUpload an image to begin conversation")

    from google.colab import files
    uploaded = files.upload()

    # grab first file
    filename = next(iter(uploaded))
    image_bytes = uploaded[filename]

    # # resize for speed
    # from PIL import Image
    # import io, base64

    # img = Image.open(io.BytesIO(image_bytes))
    # img.thumbnail((768,768))

    # buffer = io.BytesIO()
    # img.save(buffer, format="JPEG")

    # img_b64 = base64.b64encode(buffer.getvalue()).decode()
    img_b64 = prepare_image_for_llava(image_bytes)
    print("Image uploaded ✔")

    return {"image_b64": img_b64}

# def upload_image(state: AgentState):

#     print("\nUpload an image to begin conversation")

#     uploader = widgets.FileUpload(
#         accept='image/*',
#         multiple=False
#     )

#     display(uploader)

#     # wait for upload
#     import time
#     while len(uploader.value) == 0:
#         time.sleep(1)

#     fileinfo = list(uploader.value.values())[0]
#     image_bytes = fileinfo["content"]

#     # OPTIONAL: resize if large (VERY IMPORTANT FOR SPEED)
#     img = Image.open(io.BytesIO(image_bytes))
#     img.thumbnail((768,768))

#     buffer = io.BytesIO()
#     img.save(buffer, format="JPEG")

#     img_b64 = base64.b64encode(buffer.getvalue()).decode()

#     print("Image uploaded ✔")

#     return {"image_b64": img_b64}

def get_user_input(state: AgentState):

    text = input("\nAsk about the image (q to quit): ").strip()

    if text.lower() in {"q","quit","exit"}:
        return {"should_exit": True}

    return {
        "user_input": text,
        "messages": [HumanMessage(content=text)]
    }

def route_after_input(state):

    if state.get("should_exit"):
        return END

    if state.get("user_input","") == "":
        return "get_user_input"

    return "call_llava"

def call_llava(state: AgentState):

    text = state["user_input"]
    img = state["image_b64"]

    # build conversation history
    msgs = []

    for m in state["messages"][:-1]:
        role = "assistant" if isinstance(m, AIMessage) else "user"
        msgs.append({"role":role, "content":m.content})

    # current user message WITH IMAGE
    msgs.append({
        "role":"user",
        "content": text,
        "images":[img]
    })

    print("Calling LLaVA...")

    response = ollama.chat(
        model="llava",
        messages=msgs
    )

    print("LLaVA returned")

    answer = response["message"]["content"]

    return {"messages":[AIMessage(content=answer)]}

def print_response(state):

    print("\nLLaVA:")
    print(state["messages"][-1].content)

    return {}

def create_graph():

    graph = StateGraph(AgentState)

    graph.add_node("upload_image", upload_image)
    graph.add_node("get_user_input", get_user_input)
    graph.add_node("call_llava", call_llava)
    graph.add_node("print_response", print_response)

    graph.add_edge(START, "upload_image")
    graph.add_edge("upload_image", "get_user_input")

    graph.add_conditional_edges(
        "get_user_input",
        route_after_input,
        {
            "get_user_input":"get_user_input",
            "call_llava":"call_llava",
            END:END
        }
    )

    graph.add_edge("call_llava","print_response")
    graph.add_edge("print_response","get_user_input")

    return graph.compile()

graph = create_graph()

initial_state = {
    "messages":[SystemMessage(content="You are a helpful vision assistant. Answer the user's question about the provided image. Use the conversation context, and be concise but specific.")],
    "user_input":"",
    "image_b64":None,
    "should_exit":False
}

graph.invoke(initial_state)



Upload an image to begin conversation


Saving 257b0cedd7cda9dae0f35c0d5dd4a972c89b953c.jpeg to 257b0cedd7cda9dae0f35c0d5dd4a972c89b953c.jpeg
Final image size: (128, 69)
Image uploaded ✔

Ask about the image (q to quit): describe the image
Calling LLaVA...
LLaVA returned

LLaVA:
 The image depicts a stunning landscape featuring a lush valley with verdant green hills and rolling clouds in the background. At the bottom of the hill is a vibrant blue lake that reflects the surrounding scenery. The sky above the valley is partly cloudy, indicating good weather conditions. In the foreground, there's a clear view of the landscape, providing a panoramic perspective of the region. There's no text present in the image. 

Ask about the image (q to quit): what does the weather look like in the image?
Calling LLaVA...
LLaVA returned

LLaVA:
 The weather in the image appears to be sunny with partly cloudy skies, suggesting it is likely a pleasant day for outdoor activities such as hiking or sightseeing. 


In [ ]:
import base64
from typing import TypedDict, Annotated, Sequence
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, AnyMessage

import ollama
from IPython.display import display
import ipywidgets as widgets
from PIL import Image
import io
from google.colab import files
import gradio as gr


class AgentState(TypedDict):
    messages: Annotated[Sequence[AnyMessage], add_messages]
    user_input: str
    image_b64: str | None
    should_exit: bool

from PIL import Image
import io, base64

def prepare_image_for_llava(image_bytes,
                            max_dim=128,
                            max_megapixels=0.1,
                            jpeg_quality=70):
    """
    Resize + compress image so Ollama LLaVA won't crash.
    Returns base64 string.
    """

    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")

    # ---------- Step 1: limit megapixels ----------
    w, h = img.size
    megapixels = (w*h)/1_000_000

    if megapixels > max_megapixels:
        scale = (max_megapixels/megapixels) ** 0.5
        img = img.resize(
            (int(w*scale), int(h*scale)),
            Image.LANCZOS
        )

    # ---------- Step 2: limit dimension ----------
    img.thumbnail((max_dim, max_dim), Image.LANCZOS)

    # ---------- Step 3: compress JPEG ----------
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=jpeg_quality, optimize=True)

    # ---------- Step 4: convert to base64 ----------
    img_b64 = base64.b64encode(buffer.getvalue()).decode()

    print(f"Final image size: {img.size}")

    return img_b64

def route_after_input(state):

    if state.get("should_exit"):
        return END

    if state.get("user_input","") == "":
        return "get_user_input"

    return "call_llava"

def call_llava(state: AgentState):

    text = state["user_input"]
    img = state["image_b64"]

    # build conversation history
    msgs = []

    for m in state["messages"][:-1]:
        role = "assistant" if isinstance(m, AIMessage) else "user"
        msgs.append({"role":role, "content":m.content})

    # current user message WITH IMAGE
    msgs.append({
        "role":"user",
        "content": text,
        "images":[img]
    })

    response = ollama.chat(
        model="llava:7b-v1.5-q4_0",
        messages=msgs
    )

    answer = response["message"]["content"]

    return {"messages":[AIMessage(content=answer)]}

def create_graph():

    graph = StateGraph(AgentState)

    graph.add_node("call_llava", call_llava)

    graph.add_edge(START, "call_llava")
    graph.add_edge("call_llava", END)

    return graph.compile()

graph = create_graph()

# Global conversation state
state = {
    "messages":[
        SystemMessage(
            content="You are a helpful vision assistant. Answer questions about the image clearly and concisely."
        )
    ],
    "user_input":"",
    "image_b64":None,
    "should_exit":False
}

def chat_with_image(user_message, image, chat_history):

    global state

    # If image uploaded, process it
    if image is not None:
        buffered = io.BytesIO()
        image.save(buffered, format="JPEG")
        img_b64 = prepare_image_for_llava(buffered.getvalue())
        state["image_b64"] = img_b64

    if state["image_b64"] is None:
        return chat_history + [("Please upload an image first.", "")]

    # Update state
    state["user_input"] = user_message
    state["messages"].append(HumanMessage(content=user_message))

    try:
        result = graph.invoke(state)
        assistant_reply = result["messages"][-1].content

    except Exception as e:
        assistant_reply = f"Error: {str(e)}"

    state["messages"].append(AIMessage(content=assistant_reply))

    chat_history.append((user_message, assistant_reply))

    return chat_history


with gr.Blocks() as demo:
    gr.Markdown("## 🖼️ Vision-Language Chat Agent (LLaVA + LangGraph)")

    with gr.Row():
        image_input = gr.Image(type="pil", label="Upload Image")

    chatbot = gr.Chatbot()

    msg = gr.Textbox(label="Ask a question about the image")
    send_btn = gr.Button("Send")

    send_btn.click(
        chat_with_image,
        inputs=[msg, image_input, chatbot],
        outputs=chatbot
    )

demo.launch(share=True)


/tmp/ipython-input-334/2414101854.py:163: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipython-input-334/2414101854.py:163: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0a5e13d424eab9b03c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Exercise 2

In [14]:
!pip install opencv-python

In [7]:
import cv2
import base64
import io
from PIL import Image
import ollama

# ---------- IMAGE PREP (reuse your safe resize idea) ----------

def prepare_image_for_llava(frame,
                            max_dim=256,
                            jpeg_quality=70):

    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    img.thumbnail((max_dim, max_dim), Image.LANCZOS)

    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=jpeg_quality)

    img_b64 = base64.b64encode(buffer.getvalue()).decode()

    return img_b64


# ---------- FRAME EXTRACTION ----------

def extract_frames(video_path, seconds=2):

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    interval = int(fps * seconds)

    frames = []
    timestamps = []

    frame_num = 0

    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        if frame_num % interval == 0:
            frames.append(frame)
            timestamps.append(frame_num / fps)

        frame_num += 1

    cap.release()

    return frames, timestamps


# ---------- LLaVA PERSON DETECTOR ----------

def detect_person(frame):

    img_b64 = prepare_image_for_llava(frame)

    response = ollama.chat(
        model="llava:7b-v1.5-q4_0",
        messages=[{
            "role": "user",
            "content": "Answer ONLY yes or no. Is there a person visible in this scene?",
            "images": [img_b64]
        }]
    )

    answer = response["message"]["content"].lower()

    return "yes" in answer


# ---------- SURVEILLANCE ANALYSIS ----------

def analyze_video(video_path):

    frames, timestamps = extract_frames(video_path, seconds=2)

    print(f"Extracted {len(frames)} frames")

    events = []

    person_present = False

    for i, frame in enumerate(frames):

        time = timestamps[i]

        print(f"Analyzing frame {i} at {time:.1f}s")

        detected = detect_person(frame)

        # person enters
        if detected and not person_present:
            events.append(("enter", time))
            person_present = True

        # person exits
        elif not detected and person_present:
            events.append(("exit", time))
            person_present = False

    return events


# ---------- RUN ----------

video_path = "/content/walking.mov"

events = analyze_video(video_path)

print("\nDetected Events:")
for event, t in events:
    print(f"Person {event} at {t:.1f} seconds")

Extracted 63 frames
Analyzing frame 0 at 0.0s
Analyzing frame 1 at 2.0s
Analyzing frame 2 at 3.9s
Analyzing frame 3 at 5.9s
Analyzing frame 4 at 7.9s
Analyzing frame 5 at 9.8s
Analyzing frame 6 at 11.8s
Analyzing frame 7 at 13.8s
Analyzing frame 8 at 15.7s
Analyzing frame 9 at 17.7s
Analyzing frame 10 at 19.7s
Analyzing frame 11 at 21.7s
Analyzing frame 12 at 23.6s
Analyzing frame 13 at 25.6s
Analyzing frame 14 at 27.6s
Analyzing frame 15 at 29.5s
Analyzing frame 16 at 31.5s
Analyzing frame 17 at 33.5s
Analyzing frame 18 at 35.4s
Analyzing frame 19 at 37.4s
Analyzing frame 20 at 39.4s
Analyzing frame 21 at 41.3s
Analyzing frame 22 at 43.3s
Analyzing frame 23 at 45.3s
Analyzing frame 24 at 47.2s
Analyzing frame 25 at 49.2s
Analyzing frame 26 at 51.2s
Analyzing frame 27 at 53.1s
Analyzing frame 28 at 55.1s
Analyzing frame 29 at 57.1s
Analyzing frame 30 at 59.0s
Analyzing frame 31 at 61.0s
Analyzing frame 32 at 63.0s
Analyzing frame 33 at 65.0s
Analyzing frame 34 at 66.9s
Analyzing frame 

In [20]:
import cv2
import base64
import io
from PIL import Image
import ollama

# ---------- IMAGE PREP (reuse your safe resize idea) ----------

def prepare_image_for_llava(frame,
                            max_dim=256,
                            jpeg_quality=70):

    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    img.thumbnail((max_dim, max_dim), Image.LANCZOS)

    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=jpeg_quality)

    img_b64 = base64.b64encode(buffer.getvalue()).decode()

    return img_b64


# ---------- FRAME EXTRACTION ----------

def extract_frames(video_path, seconds=2):

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    interval = int(fps * seconds)

    frames = []
    timestamps = []

    frame_num = 0

    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        if frame_num % interval == 0:
            frames.append(frame)
            timestamps.append(frame_num / fps)
            print(f"Extracted frame {len(frames)} ")

        frame_num += 1

    cap.release()

    return frames, timestamps


# ---------- LLaVA PERSON DETECTOR ----------

def detect_person(frame):

    img_b64 = prepare_image_for_llava(frame)

    response = ollama.chat(
        model="llava:7b-v1.5-q4_0",
        messages=[{
            "role": "user",
            "content": "Answer ONLY yes or no. Is there a person visible in this scene?",
            "images": [img_b64]
        }]
    )

    answer = response["message"]["content"].lower()

    return "yes" in answer


# ---------- SURVEILLANCE ANALYSIS ----------

def analyze_video(video_path):

    frames, timestamps = extract_frames(video_path, seconds=2)

    print(f"Extracted {len(frames)} frames\n")

    table_rows = []

    person_present = False

    for i, frame in enumerate(frames):

        time = timestamps[i]

        print(f"Analyzing frame {i} at {time:.1f}s")

        detected = detect_person(frame)

        event = ""

        # person enters
        if detected and not person_present:
            event = "ENTER"
            person_present = True

        # person exits
        elif not detected and person_present:
            event = "EXIT"
            person_present = False

        table_rows.append({
            "frame": i,
            "timestamp": round(time, 1),
            "person": "Yes" if detected else "No",
            "event": event
        })

    return table_rows


def print_results_table(rows):

    print("\nFinal Surveillance Table\n")

    print(f"{'Frame':<10}{'Timestamp(s)':<15}{'Person?':<10}{'Event'}")
    print("-"*45)

    for r in rows:
        print(f"{r['frame']:<10}{r['timestamp']:<15}{r['person']:<10}{r['event']}")


# ---------- RUN ----------

video_path = "/content/walking.mov"

rows = analyze_video(video_path)

print_results_table(rows)

Extracted frame 1 
Extracted frame 2 
Extracted frame 3 
Extracted frame 4 
Extracted frame 5 
Extracted frame 6 
Extracted frame 7 
Extracted frame 8 
Extracted frame 9 
Extracted frame 10 
Extracted frame 11 
Extracted frame 12 
Extracted frame 13 
Extracted frame 14 
Extracted frame 15 
Extracted frame 16 
Extracted frame 17 
Extracted frame 18 
Extracted frame 19 
Extracted frame 20 
Extracted frame 21 
Extracted frame 22 
Extracted frame 23 
Extracted frame 24 
Extracted frame 25 
Extracted frame 26 
Extracted frame 27 
Extracted frame 28 
Extracted frame 29 
Extracted frame 30 
Extracted frame 31 
Extracted frame 32 
Extracted frame 33 
Extracted frame 34 
Extracted frame 35 
Extracted frame 36 
Extracted frame 37 
Extracted frame 38 
Extracted frame 39 
Extracted frame 40 
Extracted frame 41 
Extracted frame 42 
Extracted frame 43 
Extracted frame 44 
Extracted frame 45 
Extracted frame 46 
Extracted frame 47 
Extracted frame 48 
Extracted frame 49 
Extracted frame 50 
Extracted